In [ ]:
!pip install -q transformers sentencepiece torch requests

In [ ]:

import os
import json
import shutil
import logging
import requests

from dataclasses import dataclass, asdict, field
from typing import List

from transformers import pipeline

In [ ]:
import torch

device = 0 if torch.cuda.is_available() else -1

print("Device:", "GPU" if device == 0 else "CPU")

generator = pipeline(
    "text-generation",
    model="google/flan-t5-small",
    device=device
)

print("Free local AI model loaded successfully!")

Device: CPU


model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


...

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ern

Free local AI model loaded successfully!


In [ ]:

@dataclass
class WorkflowState:

    topic: str

    retrieved_chunks: List[str] = field(
        default_factory=list
    )

    extracted_points: List[str] = field(
        default_factory=list
    )

    synthesis_text: str = ""

    final_report: str = ""

    current_step: str = ""


print("WorkflowState created!")

WorkflowState created!


In [ ]:

BASE_DIR = "day28_checkpoints"

os.makedirs(BASE_DIR, exist_ok=True)


def get_run_dir(topic):

    safe_topic = (
        topic.lower()
        .replace(" ", "_")
        .replace("/", "_")
    )

    path = os.path.join(
        BASE_DIR,
        safe_topic
    )

    os.makedirs(path, exist_ok=True)

    return path


def checkpoint_path(topic, step):

    return os.path.join(
        get_run_dir(topic),
        f"{step}.json"
    )


def save_checkpoint(state, step):

    state.current_step = step

    filename = checkpoint_path(
        state.topic,
        step
    )

    with open(
        filename,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            asdict(state),
            f,
            indent=4,
            ensure_ascii=False
        )

    print(
        f"💾 Checkpoint saved: {filename}"
    )


def load_checkpoint(topic, step):

    filename = checkpoint_path(
        topic,
        step
    )

    if not os.path.exists(filename):
        return None

    with open(
        filename,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    return WorkflowState(**data)

In [ ]:

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s"
)

logger = logging.getLogger(
    "Day28Workflow"
)


def save_partial_results(
    state,
    step,
    error
):

    filename = os.path.join(
        get_run_dir(state.topic),
        f"partial_results_{step}.json"
    )

    data = asdict(state)

    data["failed_step"] = step
    data["error"] = str(error)

    with open(
        filename,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            data,
            f,
            indent=4,
            ensure_ascii=False
        )

    print(
        f"⚠️ Partial results saved: {filename}"
    )

In [ ]:

def search_sources(topic):

    print(
        f"🔎 Searching free sources for: {topic}"
    )

    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "format": "json",
        "list": "search",
        "srsearch": topic,
        "srlimit": 5
    }

    response = requests.get(
        url,
        params=params,
        timeout=20
    )

    response.raise_for_status()

    data = response.json()

    chunks = []

    for result in data["query"]["search"]:

        title = result["title"]

        page_url = (
            "https://en.wikipedia.org/wiki/"
            + title.replace(" ", "_")
        )

        chunk = (
            f"Source: {title}\n"
            f"URL: {page_url}\n"
            f"Summary: {result['snippet']}"
        )

        chunks.append(chunk)

    return chunks

In [ ]:

def extract_key_points(chunks):

    print("🧠 Extracting key points...")

    points = []

    for chunk in chunks:

        prompt = f"""
Extract 3 important factual points
from this research source.

SOURCE:
{chunk}

Return only short bullet points.
"""

        result = generator(
            prompt,
            max_new_tokens=80,
            do_sample=False
        )[0]["generated_text"]

        points.append(result.strip())

    return points

In [ ]:

def synthesise_findings(points):

    print("📝 Synthesising findings...")

    text = "\n".join(
        f"- {p}"
        for p in points
    )

    prompt = f"""
Combine these research findings into
one clear synthesis.

FINDINGS:
{text}

Write a concise research synthesis
covering major findings, benefits,
challenges and future direction.
"""

    result = generator(
        prompt,
        max_new_tokens=250,
        do_sample=False
    )[0]["generated_text"]

    return result.strip()

In [ ]:

def format_report(synthesis):

    print("📄 Formatting final report...")

    prompt = f"""
Create a professional research report
from this synthesis.

SYNTHESIS:
{synthesis}

Use these sections:

Executive Summary
Introduction
Key Findings
Benefits and Opportunities
Challenges
Future Outlook
Conclusion

Keep the report concise and structured.
"""

    result = generator(
        prompt,
        max_new_tokens=400,
        do_sample=False
    )[0]["generated_text"]

    return result.strip()

In [ ]:

def run_workflow(
    topic,
    simulate_failure=False
):

    print("\n")
    print("=" * 60)
    print("🚀 WORKFLOW STARTED")
    print("Topic:", topic)
    print("=" * 60)

    state = WorkflowState(
        topic=topic
    )

    # =================================
    # STEP 1
    # =================================

    try:

        checkpoint = load_checkpoint(
            topic,
            "step1"
        )

        if checkpoint:

            state = checkpoint

            print(
                "⏭️ Step 1 skipped - checkpoint found"
            )

        else:

            print("\nSTEP 1: SEARCH SOURCES")

            state.retrieved_chunks = (
                search_sources(topic)
            )

            save_checkpoint(
                state,
                "step1"
            )

    except Exception as e:

        logger.error(
            f"Step 1 failed: {e}"
        )

        save_partial_results(
            state,
            "step1",
            e
        )

        return state


    # =================================
    # STEP 2
    # =================================

    try:

        checkpoint = load_checkpoint(
            topic,
            "step2"
        )

        if checkpoint:

            state = checkpoint

            print(
                "⏭️ Step 2 skipped - checkpoint found"
            )

        else:

            print("\nSTEP 2: EXTRACT KEY POINTS")

            state.extracted_points = (
                extract_key_points(
                    state.retrieved_chunks
                )
            )

            save_checkpoint(
                state,
                "step2"
            )

    except Exception as e:

        logger.error(
            f"Step 2 failed: {e}"
        )

        save_partial_results(
            state,
            "step2",
            e
        )

        return state


    # =================================
    # STEP 3
    # =================================

    try:

        checkpoint = load_checkpoint(
            topic,
            "step3"
        )

        if checkpoint:

            state = checkpoint

            print(
                "⏭️ Step 3 skipped - checkpoint found"
            )

        else:

            print("\nSTEP 3: SYNTHESISE FINDINGS")

            # Intentional crash
            if simulate_failure:

                raise Exception(
                    "Intentional Step 3 failure "
                    "for resume testing"
                )

            state.synthesis_text = (
                synthesise_findings(
                    state.extracted_points
                )
            )

            save_checkpoint(
                state,
                "step3"
            )

    except Exception as e:

        logger.error(
            f"Step 3 failed: {e}"
        )

        save_partial_results(
            state,
            "step3",
            e
        )

        return state


    # =================================
    # STEP 4
    # =================================

    try:

        checkpoint = load_checkpoint(
            topic,
            "step4"
        )

        if checkpoint:

            state = checkpoint

            print(
                "⏭️ Step 4 skipped - checkpoint found"
            )

        else:

            print("\nSTEP 4: FORMAT REPORT")

            state.final_report = (
                format_report(
                    state.synthesis_text
                )
            )

            save_checkpoint(
                state,
                "step4"
            )

    except Exception as e:

        logger.error(
            f"Step 4 failed: {e}"
        )

        save_partial_results(
            state,
            "step4",
            e
        )

        return state


    print("\n")
    print("=" * 60)
    print("✅ WORKFLOW COMPLETED")
    print("=" * 60)

    return state

In [45]:

topic = "Generative AI in Education"

state = run_workflow(topic)

print("\nFINAL REPORT\n")
print(state.final_report)

ERROR:Day28Workflow:Step 1 failed: 403 Client Error: Forbidden for url: https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch=Generative+AI+in+Education&srlimit=5




🚀 WORKFLOW STARTED
Topic: Generative AI in Education

STEP 1: SEARCH SOURCES
🔎 Searching free sources for: Generative AI in Education
⚠️ Partial results saved: day28_checkpoints/generative_ai_in_education/partial_results_step1.json

FINAL REPORT




In [46]:

topic = "Artificial Intelligence in Healthcare"

run_dir = get_run_dir(topic)

if os.path.exists(run_dir):

    shutil.rmtree(run_dir)

print("Old checkpoints removed.")

Old checkpoints removed.


In [47]:

state = run_workflow(
    topic,
    simulate_failure=True
)

ERROR:Day28Workflow:Step 1 failed: 403 Client Error: Forbidden for url: https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch=Artificial+Intelligence+in+Healthcare&srlimit=5




🚀 WORKFLOW STARTED
Topic: Artificial Intelligence in Healthcare

STEP 1: SEARCH SOURCES
🔎 Searching free sources for: Artificial Intelligence in Healthcare
⚠️ Partial results saved: day28_checkpoints/artificial_intelligence_in_healthcare/partial_results_step1.json


In [48]:

state = run_workflow(
    topic,
    simulate_failure=False
)

ERROR:Day28Workflow:Step 1 failed: 403 Client Error: Forbidden for url: https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch=Artificial+Intelligence+in+Healthcare&srlimit=5




🚀 WORKFLOW STARTED
Topic: Artificial Intelligence in Healthcare

STEP 1: SEARCH SOURCES
🔎 Searching free sources for: Artificial Intelligence in Healthcare
⚠️ Partial results saved: day28_checkpoints/artificial_intelligence_in_healthcare/partial_results_step1.json


In [49]:
print(state.final_report)

In [50]:

topics = [
    "Generative AI in Education",
    "Artificial Intelligence in Healthcare",
    "Renewable Energy Technology"
]

results = {}

for topic in topics:

    print("\n")
    print("#" * 70)
    print("TOPIC:", topic)
    print("#" * 70)

    state = run_workflow(topic)

    results[topic] = state.final_report

    print("\nFINAL REPORT:\n")
    print(state.final_report)

ERROR:Day28Workflow:Step 1 failed: 403 Client Error: Forbidden for url: https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch=Generative+AI+in+Education&srlimit=5




######################################################################
TOPIC: Generative AI in Education
######################################################################


🚀 WORKFLOW STARTED
Topic: Generative AI in Education

STEP 1: SEARCH SOURCES
🔎 Searching free sources for: Generative AI in Education
⚠️ Partial results saved: day28_checkpoints/generative_ai_in_education/partial_results_step1.json

FINAL REPORT:




######################################################################
TOPIC: Artificial Intelligence in Healthcare
######################################################################


🚀 WORKFLOW STARTED
Topic: Artificial Intelligence in Healthcare

STEP 1: SEARCH SOURCES
🔎 Searching free sources for: Artificial Intelligence in Healthcare


ERROR:Day28Workflow:Step 1 failed: 403 Client Error: Forbidden for url: https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch=Artificial+Intelligence+in+Healthcare&srlimit=5
ERROR:Day28Workflow:Step 1 failed: 403 Client Error: Forbidden for url: https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch=Renewable+Energy+Technology&srlimit=5


⚠️ Partial results saved: day28_checkpoints/artificial_intelligence_in_healthcare/partial_results_step1.json

FINAL REPORT:




######################################################################
TOPIC: Renewable Energy Technology
######################################################################


🚀 WORKFLOW STARTED
Topic: Renewable Energy Technology

STEP 1: SEARCH SOURCES
🔎 Searching free sources for: Renewable Energy Technology
⚠️ Partial results saved: day28_checkpoints/renewable_energy_technology/partial_results_step1.json

FINAL REPORT:




In [51]:

for index, (topic, report) in enumerate(
    results.items(),
    start=1
):

    filename = f"research_report_{index}.txt"

    with open(
        filename,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            f"TOPIC: {topic}\n\n"
        )

        f.write(report)

    print(
        f"✅ Saved: {filename}"
    )

✅ Saved: research_report_1.txt
✅ Saved: research_report_2.txt
✅ Saved: research_report_3.txt


In [52]:

def report_stats(report):

    words = len(report.split())

    sections = [
        "Executive Summary",
        "Introduction",
        "Key Findings",
        "Benefits",
        "Challenges",
        "Future Outlook",
        "Conclusion"
    ]

    found_sections = 0

    for section in sections:

        if section.lower() in report.lower():
            found_sections += 1

    return words, found_sections


print(
    "\n"
    + "=" * 70
)

print("REPORT COMPARISON")

print("=" * 70)

for topic, report in results.items():

    words, sections = report_stats(report)

    print("\nTopic:", topic)
    print("Word count:", words)
    print(
        "Expected sections found:",
        sections
    )


REPORT COMPARISON

Topic: Generative AI in Education
Word count: 0
Expected sections found: 0

Topic: Artificial Intelligence in Healthcare
Word count: 0
Expected sections found: 0

Topic: Renewable Energy Technology
Word count: 0
Expected sections found: 0


In [53]:

print("\n📁 CHECKPOINT STRUCTURE\n")

for root, dirs, files in os.walk(
    BASE_DIR
):

    level = root.replace(
        BASE_DIR,
        ""
    ).count(os.sep)

    indent = "  " * level

    print(
        indent
        + os.path.basename(root)
        + "/"
    )

    for file in files:

        print(
            indent
            + "  📄 "
            + file
        )


📁 CHECKPOINT STRUCTURE

day28_checkpoints/
  renewable_energy_technology/
    📄 partial_results_step1.json
  generative_ai_in_education/
    📄 partial_results_step1.json
  artificial_intelligence_in_healthcare/
    📄 partial_results_step1.json
